<h1>Tidy data</h1>
            
<small>Source: <a href="https://github.com/tidyverse/tidyr/blob/pkgdown-v1.3.1/vignettes/tidy-data.Rmd"><code>vignettes/tidy-data.Rmd</code></a></small>
<div><code>tidy-data.Rmd</code></div>


<p>(This is an informal and code heavy version of the full <a href="https://vita.had.co.nz/papers/tidy-data.html">tidy data paper</a>. Please refer to that for more details.)</p>

# <h2 id="data-tidying">Data tidying<a href="#data-tidying"></a></h2>

<p>It is often said that 80% of data analysis is spent on the cleaning and preparing data. And it’s not just a first step, but it must be repeated many times over the course of analysis as new problems come to light or new data is collected. To get a handle on the problem, this paper focuses on a small, but important, aspect of data cleaning that I call data <strong>tidying</strong>: structuring datasets to facilitate analysis.</p>

<p>The principles of tidy data provide a standard way to organise data values within a dataset. A standard makes initial data cleaning easier because you don’t need to start from scratch and reinvent the wheel every time. The tidy data standard has been designed to facilitate initial exploration and analysis of the data, and to simplify the development of data analysis tools that work well together. Current tools often require translation. You have to spend time munging the output from one tool so you can input it into another. Tidy datasets and tidy tools work hand in hand to make data analysis easier, allowing you to focus on the interesting domain problem, not on the uninteresting logistics of data.</p>

# <h2 id="defining">Defining tidy data</h2>

<blockquote>
<p>Happy families are all alike; every unhappy family is unhappy in its own way — Leo Tolstoy</p>
</blockquote>

<p>Like families, tidy datasets are all alike but every messy dataset is messy in its own way. Tidy datasets provide a standardized way to link the structure of a dataset (its physical layout) with its semantics (its meaning). In this section, I’ll provide some standard vocabulary for describing the structure and semantics of a dataset, and then use those definitions to define tidy data.</p>


## <h3 id="data-structure">Data structure</h3>

<p>Most statistical datasets are data frames made up of <strong>rows</strong> and <strong>columns</strong>. The columns are almost always labeled and the rows are sometimes labeled. The following code provides some data about an imaginary classroom in a format commonly seen in the wild. The table has three columns and four rows, and both rows and columns are labeled.</p>

In [1]:
library(tibble)

classroom <- tribble(
  ~name,    ~quiz1, ~quiz2, ~test1,
  "Billy",  NA,     "D",    "C",
  "Suzy",   "F",    NA,     NA,
  "Lionel", "B",    "C",    "B",
  "Jenny",  "A",    "A",    "B"
  )

classroom

name,quiz1,quiz2,test1
<chr>,<chr>,<chr>,<chr>
Billy,NA,D,C
Suzy,F,NA,NA
Lionel,B,C,B
Jenny,A,A,B


<p>There are many ways to structure the same underlying data. The following table shows the same data as above, but the rows and columns have been transposed.</p>

In [2]:
tribble(
  ~assessment, ~Billy, ~Suzy, ~Lionel, ~Jenny,
  "quiz1",     NA,     "F",   "B",     "A",
  "quiz2",     "D",    NA,    "C",     "A",
  "test1",     "C",    NA,    "B",     "B"
  )

assessment,Billy,Suzy,Lionel,Jenny
<chr>,<chr>,<chr>,<chr>,<chr>
quiz1,NA,F,B,A
quiz2,D,NA,C,A
test1,C,NA,B,B


<p>The data is the same, but the layout is different. Our vocabulary of rows and columns is simply not rich enough to describe why the two tables represent the same data. In addition to appearance, we need a way to describe the underlying semantics, or meaning, of the values displayed in the table.</p>

## <h3 id="data-semantics">Data semantics</h3>

<p>A dataset is a collection of <strong>values</strong>, usually either numbers (if quantitative) or strings (if qualitative). Values are organised in two ways. Every value belongs to a <strong>variable</strong> and an <strong>observation</strong>. A variable contains all values that measure the same underlying attribute (like height, temperature, duration) across units. An observation contains all values measured on the same unit (like a person, or a day, or a race) across attributes.</p>

<p>A tidy version of the classroom data looks like this: (you’ll learn how the functions work a little later)</p>

In [3]:
library(tidyr)
library(dplyr)


Caricamento pacchetto: ‘dplyr’


I seguenti oggetti sono mascherati da ‘package:stats’:

    filter, lag


I seguenti oggetti sono mascherati da ‘package:base’:

    intersect, setdiff, setequal, union




In [4]:
classroom2 <- classroom |> 
  pivot_longer(quiz1:test1, names_to = "assessment", values_to = "grade") |> 
  arrange(name, assessment)

classroom2

name,assessment,grade
<chr>,<chr>,<chr>
Billy,quiz1,NA
Billy,quiz2,D
Billy,test1,C
Jenny,quiz1,A
Jenny,quiz2,A
Jenny,test1,B
Lionel,quiz1,B
Lionel,quiz2,C
Lionel,test1,B


<p>This makes the values, variables, and observations more clear. The dataset contains 36 values representing three variables and 12 observations. The variables are:</p>

<ol>
<li><p><code>name</code>, with four possible values (Billy, Suzy, Lionel, and Jenny).</p></li>
<li><p><code>assessment</code>, with three possible values (quiz1, quiz2, and test1).</p></li>
<li><p><code>grade</code>, with five or six values depending on how you think of the missing value (A, B, C, D, F, NA).</p></li>
</ol>

<p>The tidy data frame explicitly tells us the definition of an observation. In this classroom, every combination of <code>name</code> and <code>assessment</code> is a single measured observation. The dataset also informs us of missing values, which can and do have meaning. Billy was absent for the first quiz, but tried to salvage his grade. Suzy failed the first quiz, so she decided to drop the class. To calculate Billy’s final grade, we might replace this missing value with an F (or he might get a second chance to take the quiz). However, if we want to know the class average for Test 1, dropping Suzy’s structural missing value would be more appropriate than imputing a new value.</p>

<p>For a given dataset, it’s usually easy to figure out what are observations and what are variables, but it is surprisingly difficult to precisely define variables and observations in general. For example, if the columns in the classroom data were <code>height</code> and <code>weight</code> we would have been happy to call them variables. If the columns were <code>height</code> and <code>width</code>, it would be less clear cut, as we might think of height and width as values of a <code>dimension</code> variable. If the columns were <code>home phone</code> and <code>work phone</code>, we could treat these as two variables, but in a fraud detection environment we might want variables <code>phone number</code> and <code>number type</code> because the use of one phone number for multiple people might suggest fraud. A general rule of thumb is that it is easier to describe functional relationships between variables (e.g., <code>z</code> is a linear combination of <code>x</code> and <code>y</code>, <code>density</code> is the ratio of <code>weight</code> to <code>volume</code>) than between rows, and it is easier to make comparisons between groups of observations (e.g., average of group a vs.&nbsp;average of group b) than between groups of columns.</p>

<p>In a given analysis, there may be multiple levels of observation. For example, in a trial of new allergy medication we might have three observational types: demographic data collected from each person (<code>age</code>, <code>sex</code>, <code>race</code>), medical data collected from each person on each day (<code>number of sneezes</code>, <code>redness of eyes</code>), and meteorological data collected on each day (<code>temperature</code>, <code>pollen count</code>).</p>

<p>Variables may change over the course of analysis. Often the variables in the raw data are very fine grained, and may add extra modelling complexity for little explanatory gain. For example, many surveys ask variations on the same question to better get at an underlying trait. In early stages of analysis, variables correspond to questions. In later stages, you change focus to traits, computed by averaging together multiple questions. This considerably simplifies analysis because you don’t need a hierarchical model, and you can often pretend that the data is continuous, not discrete.</p>


## <h3 id="tidy-data">Tidy data</h3>

<p>Tidy data is a standard way of mapping the meaning of a dataset to its structure. A dataset is messy or tidy depending on how rows, columns and tables are matched up with observations, variables and types. In <strong>tidy data</strong>:</p>

<ol>
<li><p>Each variable is a column; each column is a variable.</p></li>
<li><p>Each observation is a row; each row is an observation.</p></li>
<li><p>Each value is a cell; each cell is a single value.</p></li>
</ol>

<p>This is Codd’s 3rd normal form, but with the constraints framed in statistical language, and the focus put on a single dataset rather than the many connected datasets common in relational databases.
<strong>Messy data</strong> is any other arrangement of the data.</p> <p>Tidy data makes it easy for an analyst or a computer to extract needed variables because it provides a standard way of structuring a dataset. Compare the different versions of the classroom data: in the messy version you need to use different strategies to extract different variables. This slows analysis and invites errors. If you consider how many data analysis operations involve all of the values in a variable (every aggregation function), you can see how important it is to extract
these values in a simple, standard way. Tidy data is particularly well suited for vectorised programming languages like R, because the layout ensures that values of different variables from the same observation are always paired.</p>

<p>While the order of variables and observations does not affect analysis, a good ordering makes it easier to scan the raw values. One way of organising variables is by their role in the analysis: are values fixed by the design of the data collection, or are they measured during the course of the experiment? Fixed variables describe the experimental design and are known in advance. Computer scientists often call fixed variables dimensions, and statisticians usually denote them with subscripts on random variables. Measured variables are what we actually
measure in the study. Fixed variables should come first, followed by measured variables, each ordered so that related variables are contiguous. Rows can then be ordered by the first variable, breaking ties with the second and subsequent (fixed) variables. This is the convention adopted by all tabular displays in this paper.</p>

# <h2 id="tidying">Tidying messy datasets</h2>

<p>Real datasets can, and often do, violate the three precepts of tidy data in almost every way imaginable. While occasionally you do get a dataset that you can start analysing immediately, this is the exception, not the rule. This section describes the five most common problems with messy datasets, along with their remedies:</p>

<ul>
<li><p>Column headers are values, not variable names.</p></li>
<li><p>Multiple variables are stored in one column.</p></li>
<li><p>Variables are stored in both rows and columns.</p></li>
<li><p>Multiple types of observational units are stored in the same table.</p></li>
<li><p>A single observational unit is stored in multiple tables.</p></li>
</ul>

<p>Surprisingly, most messy datasets, including types of messiness not explicitly described above, can be tidied with a small set of tools: pivoting (longer and wider) and separating. The following sections illustrate each problem with a real dataset that I have encountered, and show how to tidy them.</p>


## <h3 id="column-headers-are-values-not-variable-names">Column headers are values, not variable names</h3>

<p>A common type of messy dataset is tabular data designed for presentation, where variables form both the rows and columns, and column headers are values, not variable names. While I would call this arrangement messy, in some cases it can be extremely useful. It provides efficient storage for completely crossed designs, and it can lead to
extremely efficient computation if desired operations can be expressed as matrix operations.</p>

<p>The following code shows a subset of a typical dataset of this form. This dataset explores the relationship between income and religion in the US. It comes from a report produced by the Pew Research Center, an American think-tank that collects data on attitudes to topics ranging from religion to the internet, and produces many reports that contain datasets in this format.</p>

<div class="sourceCode hasCopyButton" id="cb5"><button type="button" class="btn btn-primary btn-copy-ex" aria-label="Copy to clipboard" data-toggle="tooltip" data-placement="left" data-trigger="hover" data-clipboard-copy="" data-bs-original-title="Copy to clipboard"><i class="fa fa-copy"></i></button><pre class="downlit sourceCode r"><code class="sourceCode R"><span><span class="va">relig_income</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># A tibble: 18 × 11</span></span></span>
<span><span class="co">#&gt;    religion   `&lt;$10k` `$10-20k` `$20-30k` `$30-40k` `$40-50k` `$50-75k`</span></span>
<span><span class="co">#&gt;    <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>        <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span>     <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span>     <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span>     <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span>     <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span>     <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 1</span> Agnostic        27        34        60        81        76       137</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 2</span> Atheist         12        27        37        52        35        70</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 3</span> Buddhist        27        21        30        34        33        58</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 4</span> Catholic       418       617       732       670       638      <span style="text-decoration: underline;">1</span>116</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 5</span> Don’t kno…      15        14        15        11        10        35</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 6</span> Evangelic…     575       869      <span style="text-decoration: underline;">1</span>064       982       881      <span style="text-decoration: underline;">1</span>486</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 7</span> Hindu            1         9         7         9        11        34</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 8</span> Historica…     228       244       236       238       197       223</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 9</span> Jehovah's…      20        27        24        24        21        30</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;">10</span> Jewish          19        19        25        25        30        95</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 8 more rows</span></span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 4 more variables: `$75-100k` &lt;dbl&gt;, `$100-150k` &lt;dbl&gt;,</span></span></span>
<span><span class="co">#&gt; <span style="color: #949494;">#   `&gt;150k` &lt;dbl&gt;, `Don't know/refused` &lt;dbl&gt;</span></span></span></code></pre></div>

<p>This dataset has three variables, <code>religion</code>, <code>income</code> and <code>frequency</code>. To tidy it, we need to <strong>pivot</strong> the non-variable columns into a two-column key-value pair. This action is often described as making a wide dataset longer (or taller).</p>

<p>When pivoting variables, we need to provide the name of the new key-value columns to create. After defining the columns to pivot (every column except for religion), you will need the name of the key column, which is the name of the variable defined by the values of the column headings. In this case, it’s <code>income</code>. The second argument is the name of the value column, <code>frequency</code>.</p>

<div class="sourceCode hasCopyButton" id="cb6"><button type="button" class="btn btn-primary btn-copy-ex" aria-label="Copy to clipboard" data-toggle="tooltip" data-placement="left" data-trigger="hover" data-clipboard-copy="" data-bs-original-title="Copy to clipboard"><i class="fa fa-copy"></i></button><pre class="downlit sourceCode r"><code class="sourceCode R"><span><span class="va">relig_income</span> <span class="op"><a href="../reference/pipe.html">%&gt;%</a></span> </span>
<span>  <span class="fu"><a href="../reference/pivot_longer.html">pivot_longer</a></span><span class="op">(</span><span class="op">-</span><span class="va">religion</span>, names_to <span class="op">=</span> <span class="st">"income"</span>, values_to <span class="op">=</span> <span class="st">"frequency"</span><span class="op">)</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># A tibble: 180 × 3</span></span></span>
<span><span class="co">#&gt;    religion income             frequency</span></span>
<span><span class="co">#&gt;    <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>    <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>                  <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 1</span> Agnostic &lt;$10k                     27</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 2</span> Agnostic $10-20k                   34</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 3</span> Agnostic $20-30k                   60</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 4</span> Agnostic $30-40k                   81</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 5</span> Agnostic $40-50k                   76</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 6</span> Agnostic $50-75k                  137</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 7</span> Agnostic $75-100k                 122</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 8</span> Agnostic $100-150k                109</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 9</span> Agnostic &gt;150k                     84</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;">10</span> Agnostic Don't know/refused        96</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 170 more rows</span></span></span></code></pre></div>

<p>This form is tidy because each column represents a variable and each row represents an observation, in this case a demographic unit corresponding to a combination of <code>religion</code> and <code>income</code>.</p>

<p>This format is also used to record regularly spaced observations over time. For example, the Billboard dataset shown below records the date a song first entered the billboard top 100. It has variables for <code>artist</code>, <code>track</code>, <code>date.entered</code>, <code>rank</code> and <code>week</code>. The rank in each week after it enters the top 100 is recorded in 75 columns, <code>wk1</code> to <code>wk75</code>. This form of storage is not tidy, but it is useful for data entry. It reduces duplication since otherwise each song in each week would need its own row, and song metadata like title and artist would need to be repeated. This will be discussed in more depth in <a href="#multiple-types">multiple types</a>.</p>

<div class="sourceCode hasCopyButton" id="cb7"><button type="button" class="btn btn-primary btn-copy-ex" aria-label="Copy to clipboard" data-toggle="tooltip" data-placement="left" data-trigger="hover" data-clipboard-copy="" data-bs-original-title="Copy to clipboard"><i class="fa fa-copy"></i></button><pre class="downlit sourceCode r"><code class="sourceCode R"><span><span class="va">billboard</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># A tibble: 317 × 79</span></span></span>
<span><span class="co">#&gt;    artist  track date.entered   wk1   wk2   wk3   wk4   wk5   wk6   wk7</span></span>
<span><span class="co">#&gt;    <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>   <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span> <span style="color: #949494; font-style: italic;">&lt;date&gt;</span>       <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 1</span> 2 Pac   Baby… 2000-02-26      87    82    72    77    87    94    99</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 2</span> 2Ge+her The … 2000-09-02      91    87    92    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 3</span> 3 Door… Kryp… 2000-04-08      81    70    68    67    66    57    54</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 4</span> 3 Door… Loser 2000-10-21      76    76    72    69    67    65    55</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 5</span> 504 Bo… Wobb… 2000-04-15      57    34    25    17    17    31    36</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 6</span> 98^0    Give… 2000-08-19      51    39    34    26    26    19     2</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 7</span> A*Teens Danc… 2000-07-08      97    97    96    95   100    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 8</span> Aaliyah I Do… 2000-01-29      84    62    51    41    38    35    35</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 9</span> Aaliyah Try … 2000-03-18      59    53    38    28    21    18    16</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;">10</span> Adams,… Open… 2000-08-26      76    76    74    69    68    67    61</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 307 more rows</span></span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 69 more variables: wk8 &lt;dbl&gt;, wk9 &lt;dbl&gt;, wk10 &lt;dbl&gt;, wk11 &lt;dbl&gt;,</span></span></span>
<span><span class="co">#&gt; <span style="color: #949494;">#   wk12 &lt;dbl&gt;, wk13 &lt;dbl&gt;, wk14 &lt;dbl&gt;, wk15 &lt;dbl&gt;, wk16 &lt;dbl&gt;,</span></span></span>
<span><span class="co">#&gt; <span style="color: #949494;">#   wk17 &lt;dbl&gt;, wk18 &lt;dbl&gt;, wk19 &lt;dbl&gt;, wk20 &lt;dbl&gt;, wk21 &lt;dbl&gt;,</span></span></span>
<span><span class="co">#&gt; <span style="color: #949494;">#   wk22 &lt;dbl&gt;, wk23 &lt;dbl&gt;, wk24 &lt;dbl&gt;, wk25 &lt;dbl&gt;, wk26 &lt;dbl&gt;,</span></span></span>
<span><span class="co">#&gt; <span style="color: #949494;">#   wk27 &lt;dbl&gt;, wk28 &lt;dbl&gt;, wk29 &lt;dbl&gt;, wk30 &lt;dbl&gt;, wk31 &lt;dbl&gt;,</span></span></span>
<span><span class="co">#&gt; <span style="color: #949494;">#   wk32 &lt;dbl&gt;, wk33 &lt;dbl&gt;, wk34 &lt;dbl&gt;, wk35 &lt;dbl&gt;, wk36 &lt;dbl&gt;, …</span></span></span></code></pre></div>

<p>To tidy this dataset, we first use <code><a href="https://tidyr.tidyverse.org/reference/pivot_longer.html">pivot_longer()</a></code> to make the dataset longer. We transform the columns from <code>wk1</code> to <code>wk76</code>, making a new column for their names, <code>week</code>, and a new value for their values, <code>rank</code>:</p>

<div class="sourceCode hasCopyButton" id="cb8"><button type="button" class="btn btn-primary btn-copy-ex" aria-label="Copy to clipboard" data-toggle="tooltip" data-placement="left" data-trigger="hover" data-clipboard-copy="" data-bs-original-title="Copy to clipboard"><i class="fa fa-copy"></i></button><pre class="downlit sourceCode r"><code class="sourceCode R"><span><span class="va">billboard2</span> <span class="op">&lt;-</span> <span class="va">billboard</span> <span class="op"><a href="../reference/pipe.html">%&gt;%</a></span> </span>
<span>  <span class="fu"><a href="../reference/pivot_longer.html">pivot_longer</a></span><span class="op">(</span></span>
<span>    <span class="va">wk1</span><span class="op">:</span><span class="va">wk76</span>, </span>
<span>    names_to <span class="op">=</span> <span class="st">"week"</span>, </span>
<span>    values_to <span class="op">=</span> <span class="st">"rank"</span>, </span>
<span>    values_drop_na <span class="op">=</span> <span class="cn">TRUE</span></span>
<span>  <span class="op">)</span></span>
<span><span class="va">billboard2</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># A tibble: 5,307 × 5</span></span></span>
<span><span class="co">#&gt;    artist  track                   date.entered week   rank</span></span>
<span><span class="co">#&gt;    <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>   <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>                   <span style="color: #949494; font-style: italic;">&lt;date&gt;</span>       <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 1</span> 2 Pac   Baby Don't Cry (Keep... 2000-02-26   wk1      87</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 2</span> 2 Pac   Baby Don't Cry (Keep... 2000-02-26   wk2      82</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 3</span> 2 Pac   Baby Don't Cry (Keep... 2000-02-26   wk3      72</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 4</span> 2 Pac   Baby Don't Cry (Keep... 2000-02-26   wk4      77</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 5</span> 2 Pac   Baby Don't Cry (Keep... 2000-02-26   wk5      87</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 6</span> 2 Pac   Baby Don't Cry (Keep... 2000-02-26   wk6      94</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 7</span> 2 Pac   Baby Don't Cry (Keep... 2000-02-26   wk7      99</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 8</span> 2Ge+her The Hardest Part Of ... 2000-09-02   wk1      91</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 9</span> 2Ge+her The Hardest Part Of ... 2000-09-02   wk2      87</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;">10</span> 2Ge+her The Hardest Part Of ... 2000-09-02   wk3      92</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 5,297 more rows</span></span></span></code></pre></div>

<p>Here we use <code>values_drop_na = TRUE</code> to drop any missing values from the rank column. In this data, missing values represent weeks that the song wasn’t in the charts, so can be safely dropped.</p>

<p>In this case it’s also nice to do a little cleaning, converting the week variable to a number, and figuring out the date corresponding to each week on the charts:</p>

<div class="sourceCode hasCopyButton" id="cb9"><button type="button" class="btn btn-primary btn-copy-ex" aria-label="Copy to clipboard" data-toggle="tooltip" data-placement="left" data-trigger="hover" data-clipboard-copy="" data-bs-original-title="Copy to clipboard"><i class="fa fa-copy"></i></button><pre class="downlit sourceCode r"><code class="sourceCode R"><span><span class="va">billboard3</span> <span class="op">&lt;-</span> <span class="va">billboard2</span> <span class="op"><a href="../reference/pipe.html">%&gt;%</a></span></span>
<span>  <span class="fu"><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate</a></span><span class="op">(</span></span>
<span>    week <span class="op">=</span> <span class="fu"><a href="https://rdrr.io/r/base/integer.html">as.integer</a></span><span class="op">(</span><span class="fu"><a href="https://rdrr.io/r/base/grep.html">gsub</a></span><span class="op">(</span><span class="st">"wk"</span>, <span class="st">""</span>, <span class="va">week</span><span class="op">)</span><span class="op">)</span>,</span>
<span>    date <span class="op">=</span> <span class="fu"><a href="https://rdrr.io/r/base/as.Date.html">as.Date</a></span><span class="op">(</span><span class="va">date.entered</span><span class="op">)</span> <span class="op">+</span> <span class="fl">7</span> <span class="op">*</span> <span class="op">(</span><span class="va">week</span> <span class="op">-</span> <span class="fl">1</span><span class="op">)</span>,</span>
<span>    date.entered <span class="op">=</span> <span class="cn">NULL</span></span>
<span>  <span class="op">)</span></span>
<span><span class="va">billboard3</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># A tibble: 5,307 × 5</span></span></span>
<span><span class="co">#&gt;    artist  track                    week  rank date      </span></span>
<span><span class="co">#&gt;    <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>   <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>                   <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span> <span style="color: #949494; font-style: italic;">&lt;date&gt;</span>    </span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 1</span> 2 Pac   Baby Don't Cry (Keep...     1    87 2000-02-26</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 2</span> 2 Pac   Baby Don't Cry (Keep...     2    82 2000-03-04</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 3</span> 2 Pac   Baby Don't Cry (Keep...     3    72 2000-03-11</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 4</span> 2 Pac   Baby Don't Cry (Keep...     4    77 2000-03-18</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 5</span> 2 Pac   Baby Don't Cry (Keep...     5    87 2000-03-25</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 6</span> 2 Pac   Baby Don't Cry (Keep...     6    94 2000-04-01</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 7</span> 2 Pac   Baby Don't Cry (Keep...     7    99 2000-04-08</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 8</span> 2Ge+her The Hardest Part Of ...     1    91 2000-09-02</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 9</span> 2Ge+her The Hardest Part Of ...     2    87 2000-09-09</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;">10</span> 2Ge+her The Hardest Part Of ...     3    92 2000-09-16</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 5,297 more rows</span></span></span></code></pre></div>

<p>Finally, it’s always a good idea to sort the data. We could do it by artist, track and week:</p>

<div class="sourceCode hasCopyButton" id="cb10"><button type="button" class="btn btn-primary btn-copy-ex" aria-label="Copy to clipboard" data-toggle="tooltip" data-placement="left" data-trigger="hover" data-clipboard-copy="" data-bs-original-title="Copy to clipboard"><i class="fa fa-copy"></i></button><pre class="downlit sourceCode r"><code class="sourceCode R"><span><span class="va">billboard3</span> <span class="op"><a href="../reference/pipe.html">%&gt;%</a></span> <span class="fu"><a href="https://dplyr.tidyverse.org/reference/arrange.html">arrange</a></span><span class="op">(</span><span class="va">artist</span>, <span class="va">track</span>, <span class="va">week</span><span class="op">)</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># A tibble: 5,307 × 5</span></span></span>
<span><span class="co">#&gt;    artist  track                    week  rank date      </span></span>
<span><span class="co">#&gt;    <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>   <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>                   <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span> <span style="color: #949494; font-style: italic;">&lt;date&gt;</span>    </span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 1</span> 2 Pac   Baby Don't Cry (Keep...     1    87 2000-02-26</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 2</span> 2 Pac   Baby Don't Cry (Keep...     2    82 2000-03-04</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 3</span> 2 Pac   Baby Don't Cry (Keep...     3    72 2000-03-11</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 4</span> 2 Pac   Baby Don't Cry (Keep...     4    77 2000-03-18</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 5</span> 2 Pac   Baby Don't Cry (Keep...     5    87 2000-03-25</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 6</span> 2 Pac   Baby Don't Cry (Keep...     6    94 2000-04-01</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 7</span> 2 Pac   Baby Don't Cry (Keep...     7    99 2000-04-08</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 8</span> 2Ge+her The Hardest Part Of ...     1    91 2000-09-02</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 9</span> 2Ge+her The Hardest Part Of ...     2    87 2000-09-09</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;">10</span> 2Ge+her The Hardest Part Of ...     3    92 2000-09-16</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 5,297 more rows</span></span></span></code></pre></div>

<p>Or by date and rank:</p>

<div class="sourceCode hasCopyButton" id="cb11"><button type="button" class="btn btn-primary btn-copy-ex" aria-label="Copy to clipboard" data-toggle="tooltip" data-placement="left" data-trigger="hover" data-clipboard-copy="" data-bs-original-title="Copy to clipboard"><i class="fa fa-copy"></i></button><pre class="downlit sourceCode r"><code class="sourceCode R"><span><span class="va">billboard3</span> <span class="op"><a href="../reference/pipe.html">%&gt;%</a></span> <span class="fu"><a href="https://dplyr.tidyverse.org/reference/arrange.html">arrange</a></span><span class="op">(</span><span class="va">date</span>, <span class="va">rank</span><span class="op">)</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># A tibble: 5,307 × 5</span></span></span>
<span><span class="co">#&gt;    artist   track   week  rank date      </span></span>
<span><span class="co">#&gt;    <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>    <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>  <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span> <span style="color: #949494; font-style: italic;">&lt;date&gt;</span>    </span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 1</span> Lonestar Amazed     1    81 1999-06-05</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 2</span> Lonestar Amazed     2    54 1999-06-12</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 3</span> Lonestar Amazed     3    44 1999-06-19</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 4</span> Lonestar Amazed     4    39 1999-06-26</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 5</span> Lonestar Amazed     5    38 1999-07-03</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 6</span> Lonestar Amazed     6    33 1999-07-10</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 7</span> Lonestar Amazed     7    29 1999-07-17</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 8</span> Amber    Sexual     1    99 1999-07-17</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 9</span> Lonestar Amazed     8    29 1999-07-24</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;">10</span> Amber    Sexual     2    99 1999-07-24</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 5,297 more rows</span></span></span></code></pre></div>

## <h3 id="multiple-variables-stored-in-one-column">Multiple variables stored in one column</h3>

<p>After pivoting columns, the key column is sometimes a combination of multiple underlying variable names. This happens in the <code>tb</code> (tuberculosis) dataset, shown below. This dataset comes from the World Health Organisation, and records the counts of confirmed tuberculosis cases by <code>country</code>, <code>year</code>, and demographic group.
The demographic groups are broken down by <code>sex</code> (m, f) and <code>age</code> (0-14, 15-25, 25-34, 35-44, 45-54, 55-64, unknown).</p>

<div class="sourceCode hasCopyButton" id="cb12"><button type="button" class="btn btn-primary btn-copy-ex" aria-label="Copy to clipboard" data-toggle="tooltip" data-placement="left" data-trigger="hover" data-clipboard-copy="" data-bs-original-title="Copy to clipboard"><i class="fa fa-copy"></i></button><pre class="downlit sourceCode r"><code class="sourceCode R"><span><span class="va">tb</span> <span class="op">&lt;-</span> <span class="fu"><a href="https://tibble.tidyverse.org/reference/as_tibble.html">as_tibble</a></span><span class="op">(</span><span class="fu"><a href="https://rdrr.io/r/utils/read.table.html">read.csv</a></span><span class="op">(</span><span class="st">"tb.csv"</span>, stringsAsFactors <span class="op">=</span> <span class="cn">FALSE</span><span class="op">)</span><span class="op">)</span></span>
<span><span class="va">tb</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># A tibble: 5,769 × 22</span></span></span>
<span><span class="co">#&gt;    iso2   year   m04  m514  m014 m1524 m2534 m3544 m4554 m5564   m65</span></span>
<span><span class="co">#&gt;    <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 1</span> AD     <span style="text-decoration: underline;">1</span>989    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 2</span> AD     <span style="text-decoration: underline;">1</span>990    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 3</span> AD     <span style="text-decoration: underline;">1</span>991    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 4</span> AD     <span style="text-decoration: underline;">1</span>992    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 5</span> AD     <span style="text-decoration: underline;">1</span>993    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 6</span> AD     <span style="text-decoration: underline;">1</span>994    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 7</span> AD     <span style="text-decoration: underline;">1</span>996    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>     0     0     0     4     1     0     0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 8</span> AD     <span style="text-decoration: underline;">1</span>997    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>     0     0     1     2     2     1     6</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 9</span> AD     <span style="text-decoration: underline;">1</span>998    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>     0     0     0     1     0     0     0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;">10</span> AD     <span style="text-decoration: underline;">1</span>999    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>     0     0     0     1     1     0     0</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 5,759 more rows</span></span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 11 more variables: mu &lt;int&gt;, f04 &lt;int&gt;, f514 &lt;int&gt;, f014 &lt;int&gt;,</span></span></span>
<span><span class="co">#&gt; <span style="color: #949494;">#   f1524 &lt;int&gt;, f2534 &lt;int&gt;, f3544 &lt;int&gt;, f4554 &lt;int&gt;, f5564 &lt;int&gt;,</span></span></span>
<span><span class="co">#&gt; <span style="color: #949494;">#   f65 &lt;int&gt;, fu &lt;int&gt;</span></span></span></code></pre></div>

<p>First we use <code><a href="../reference/pivot_longer.html">pivot_longer()</a></code> to gather up the non-variable columns:</p>

<div class="sourceCode hasCopyButton" id="cb13"><button type="button" class="btn btn-primary btn-copy-ex" aria-label="Copy to clipboard" data-toggle="tooltip" data-placement="left" data-trigger="hover" data-clipboard-copy="" data-bs-original-title="Copy to clipboard"><i class="fa fa-copy"></i></button><pre class="downlit sourceCode r"><code class="sourceCode R"><span><span class="va">tb2</span> <span class="op">&lt;-</span> <span class="va">tb</span> <span class="op"><a href="../reference/pipe.html">%&gt;%</a></span> </span>
<span>  <span class="fu"><a href="../reference/pivot_longer.html">pivot_longer</a></span><span class="op">(</span></span>
<span>    <span class="op">!</span><span class="fu"><a href="https://rdrr.io/r/base/c.html">c</a></span><span class="op">(</span><span class="va">iso2</span>, <span class="va">year</span><span class="op">)</span>, </span>
<span>    names_to <span class="op">=</span> <span class="st">"demo"</span>, </span>
<span>    values_to <span class="op">=</span> <span class="st">"n"</span>, </span>
<span>    values_drop_na <span class="op">=</span> <span class="cn">TRUE</span></span>
<span>  <span class="op">)</span></span>
<span><span class="va">tb2</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># A tibble: 35,750 × 4</span></span></span>
<span><span class="co">#&gt;    iso2   year demo      n</span></span>
<span><span class="co">#&gt;    <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 1</span> AD     <span style="text-decoration: underline;">1</span>996 m014      0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 2</span> AD     <span style="text-decoration: underline;">1</span>996 m1524     0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 3</span> AD     <span style="text-decoration: underline;">1</span>996 m2534     0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 4</span> AD     <span style="text-decoration: underline;">1</span>996 m3544     4</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 5</span> AD     <span style="text-decoration: underline;">1</span>996 m4554     1</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 6</span> AD     <span style="text-decoration: underline;">1</span>996 m5564     0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 7</span> AD     <span style="text-decoration: underline;">1</span>996 m65       0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 8</span> AD     <span style="text-decoration: underline;">1</span>996 f014      0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 9</span> AD     <span style="text-decoration: underline;">1</span>996 f1524     1</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;">10</span> AD     <span style="text-decoration: underline;">1</span>996 f2534     1</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 35,740 more rows</span></span></span></code></pre></div>

<p>Column headers in this format are often separated by a non-alphanumeric character (e.g.&nbsp;<code>.</code>, <code>-</code>, <code>_</code>, <code>:</code>), or have a fixed width format, like in this dataset. <code><a href="../reference/separate.html">separate()</a></code> makes it easy to split a compound variables into individual variables. You can either pass it a regular
expression to split on (the default is to split on non-alphanumeric columns), or a vector of character positions. In this case we want to split after the first character:</p>

<div class="sourceCode hasCopyButton" id="cb14"><button type="button" class="btn btn-primary btn-copy-ex" aria-label="Copy to clipboard" data-toggle="tooltip" data-placement="left" data-trigger="hover" data-clipboard-copy="" data-bs-original-title="Copy to clipboard"><i class="fa fa-copy"></i></button><pre class="downlit sourceCode r"><code class="sourceCode R"><span><span class="va">tb3</span> <span class="op">&lt;-</span> <span class="va">tb2</span> <span class="op"><a href="../reference/pipe.html">%&gt;%</a></span> </span>
<span>  <span class="fu"><a href="../reference/separate.html">separate</a></span><span class="op">(</span><span class="va">demo</span>, <span class="fu"><a href="https://rdrr.io/r/base/c.html">c</a></span><span class="op">(</span><span class="st">"sex"</span>, <span class="st">"age"</span><span class="op">)</span>, <span class="fl">1</span><span class="op">)</span></span>
<span><span class="va">tb3</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># A tibble: 35,750 × 5</span></span></span>
<span><span class="co">#&gt;    iso2   year sex   age       n</span></span>
<span><span class="co">#&gt;    <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span> <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 1</span> AD     <span style="text-decoration: underline;">1</span>996 m     014       0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 2</span> AD     <span style="text-decoration: underline;">1</span>996 m     1524      0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 3</span> AD     <span style="text-decoration: underline;">1</span>996 m     2534      0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 4</span> AD     <span style="text-decoration: underline;">1</span>996 m     3544      4</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 5</span> AD     <span style="text-decoration: underline;">1</span>996 m     4554      1</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 6</span> AD     <span style="text-decoration: underline;">1</span>996 m     5564      0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 7</span> AD     <span style="text-decoration: underline;">1</span>996 m     65        0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 8</span> AD     <span style="text-decoration: underline;">1</span>996 f     014       0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 9</span> AD     <span style="text-decoration: underline;">1</span>996 f     1524      1</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;">10</span> AD     <span style="text-decoration: underline;">1</span>996 f     2534      1</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 35,740 more rows</span></span></span></code></pre></div>

<p>Storing the values in this form resolves a problem in the original data. We want to compare rates, not counts, which means we need to know the population. In the original format, there is no easy way to add a population variable. It has to be stored in a separate table, which makes it hard to correctly match populations to counts. In tidy form, adding variables for population and rate is easy because they’re just additional columns.</p>

<p>In this case, we could also do the transformation in a single step by supplying multiple column names to <code>names_to</code> and also supplying a grouped regular expression to <code>names_pattern</code>:</p>

<div class="sourceCode hasCopyButton" id="cb15"><button type="button" class="btn btn-primary btn-copy-ex" aria-label="Copy to clipboard" data-toggle="tooltip" data-placement="left" data-trigger="hover" data-clipboard-copy="" data-bs-original-title="Copy to clipboard"><i class="fa fa-copy"></i></button><pre class="downlit sourceCode r"><code class="sourceCode R"><span><span class="va">tb</span> <span class="op"><a href="../reference/pipe.html">%&gt;%</a></span> <span class="fu"><a href="../reference/pivot_longer.html">pivot_longer</a></span><span class="op">(</span></span>
<span>  <span class="op">!</span><span class="fu"><a href="https://rdrr.io/r/base/c.html">c</a></span><span class="op">(</span><span class="va">iso2</span>, <span class="va">year</span><span class="op">)</span>, </span>
<span>  names_to <span class="op">=</span> <span class="fu"><a href="https://rdrr.io/r/base/c.html">c</a></span><span class="op">(</span><span class="st">"sex"</span>, <span class="st">"age"</span><span class="op">)</span>, </span>
<span>  names_pattern <span class="op">=</span> <span class="st">"(.)(.+)"</span>,</span>
<span>  values_to <span class="op">=</span> <span class="st">"n"</span>, </span>
<span>  values_drop_na <span class="op">=</span> <span class="cn">TRUE</span></span>
<span><span class="op">)</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># A tibble: 35,750 × 5</span></span></span>
<span><span class="co">#&gt;    iso2   year sex   age       n</span></span>
<span><span class="co">#&gt;    <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span> <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 1</span> AD     <span style="text-decoration: underline;">1</span>996 m     014       0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 2</span> AD     <span style="text-decoration: underline;">1</span>996 m     1524      0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 3</span> AD     <span style="text-decoration: underline;">1</span>996 m     2534      0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 4</span> AD     <span style="text-decoration: underline;">1</span>996 m     3544      4</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 5</span> AD     <span style="text-decoration: underline;">1</span>996 m     4554      1</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 6</span> AD     <span style="text-decoration: underline;">1</span>996 m     5564      0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 7</span> AD     <span style="text-decoration: underline;">1</span>996 m     65        0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 8</span> AD     <span style="text-decoration: underline;">1</span>996 f     014       0</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 9</span> AD     <span style="text-decoration: underline;">1</span>996 f     1524      1</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;">10</span> AD     <span style="text-decoration: underline;">1</span>996 f     2534      1</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 35,740 more rows</span></span></span></code></pre></div>

## <h3 id="variables-are-stored-in-both-rows-and-columns">Variables are stored in both rows and columns</h3>

<p>The most complicated form of messy data occurs when variables are stored in both rows and columns. The code below loads daily weather data from the Global Historical Climatology Network for one weather station (MX17004) in Mexico for five months in 2010.</p>

<div class="sourceCode hasCopyButton" id="cb16"><button type="button" class="btn btn-primary btn-copy-ex" aria-label="Copy to clipboard" data-toggle="tooltip" data-placement="left" data-trigger="hover" data-clipboard-copy="" data-bs-original-title="Copy to clipboard"><i class="fa fa-copy"></i></button><pre class="downlit sourceCode r"><code class="sourceCode R"><span><span class="va">weather</span> <span class="op">&lt;-</span> <span class="fu"><a href="https://tibble.tidyverse.org/reference/as_tibble.html">as_tibble</a></span><span class="op">(</span><span class="fu"><a href="https://rdrr.io/r/utils/read.table.html">read.csv</a></span><span class="op">(</span><span class="st">"weather.csv"</span>, stringsAsFactors <span class="op">=</span> <span class="cn">FALSE</span><span class="op">)</span><span class="op">)</span></span>
<span><span class="va">weather</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># A tibble: 22 × 35</span></span></span>
<span><span class="co">#&gt;    id      year month element    d1    d2    d3    d4    d5    d6    d7</span></span>
<span><span class="co">#&gt;    <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>  <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>   <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 1</span> MX170…  <span style="text-decoration: underline;">2</span>010     1 tmax       <span style="color: #BB0000;">NA</span>  <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>      <span style="color: #BB0000;">NA</span>  <span style="color: #BB0000;">NA</span>      <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 2</span> MX170…  <span style="text-decoration: underline;">2</span>010     1 tmin       <span style="color: #BB0000;">NA</span>  <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>      <span style="color: #BB0000;">NA</span>  <span style="color: #BB0000;">NA</span>      <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 3</span> MX170…  <span style="text-decoration: underline;">2</span>010     2 tmax       <span style="color: #BB0000;">NA</span>  27.3  24.1    <span style="color: #BB0000;">NA</span>  <span style="color: #BB0000;">NA</span>      <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 4</span> MX170…  <span style="text-decoration: underline;">2</span>010     2 tmin       <span style="color: #BB0000;">NA</span>  14.4  14.4    <span style="color: #BB0000;">NA</span>  <span style="color: #BB0000;">NA</span>      <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 5</span> MX170…  <span style="text-decoration: underline;">2</span>010     3 tmax       <span style="color: #BB0000;">NA</span>  <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>      <span style="color: #BB0000;">NA</span>  32.1    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 6</span> MX170…  <span style="text-decoration: underline;">2</span>010     3 tmin       <span style="color: #BB0000;">NA</span>  <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>      <span style="color: #BB0000;">NA</span>  14.2    <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 7</span> MX170…  <span style="text-decoration: underline;">2</span>010     4 tmax       <span style="color: #BB0000;">NA</span>  <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>      <span style="color: #BB0000;">NA</span>  <span style="color: #BB0000;">NA</span>      <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 8</span> MX170…  <span style="text-decoration: underline;">2</span>010     4 tmin       <span style="color: #BB0000;">NA</span>  <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>      <span style="color: #BB0000;">NA</span>  <span style="color: #BB0000;">NA</span>      <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 9</span> MX170…  <span style="text-decoration: underline;">2</span>010     5 tmax       <span style="color: #BB0000;">NA</span>  <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>      <span style="color: #BB0000;">NA</span>  <span style="color: #BB0000;">NA</span>      <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;">10</span> MX170…  <span style="text-decoration: underline;">2</span>010     5 tmin       <span style="color: #BB0000;">NA</span>  <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span>      <span style="color: #BB0000;">NA</span>  <span style="color: #BB0000;">NA</span>      <span style="color: #BB0000;">NA</span>    <span style="color: #BB0000;">NA</span></span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 12 more rows</span></span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 24 more variables: d8 &lt;dbl&gt;, d9 &lt;lgl&gt;, d10 &lt;dbl&gt;, d11 &lt;dbl&gt;,</span></span></span>
<span><span class="co">#&gt; <span style="color: #949494;">#   d12 &lt;lgl&gt;, d13 &lt;dbl&gt;, d14 &lt;dbl&gt;, d15 &lt;dbl&gt;, d16 &lt;dbl&gt;, d17 &lt;dbl&gt;,</span></span></span>
<span><span class="co">#&gt; <span style="color: #949494;">#   d18 &lt;lgl&gt;, d19 &lt;lgl&gt;, d20 &lt;lgl&gt;, d21 &lt;lgl&gt;, d22 &lt;lgl&gt;, d23 &lt;dbl&gt;,</span></span></span>
<span><span class="co">#&gt; <span style="color: #949494;">#   d24 &lt;lgl&gt;, d25 &lt;dbl&gt;, d26 &lt;dbl&gt;, d27 &lt;dbl&gt;, d28 &lt;dbl&gt;, d29 &lt;dbl&gt;,</span></span></span>
<span><span class="co">#&gt; <span style="color: #949494;">#   d30 &lt;dbl&gt;, d31 &lt;dbl&gt;</span></span></span></code></pre></div>

<p>It has variables in individual columns (<code>id</code>, <code>year</code>, <code>month</code>), spread across columns (<code>day</code>, d1-d31) and across rows (<code>tmin</code>, <code>tmax</code>) (minimum and maximum temperature). Months with fewer than 31 days have structural missing values for the last day(s) of the month.</p>

<p>To tidy this dataset we first use pivot_longer to gather the day columns:</p>

<div class="sourceCode hasCopyButton" id="cb17"><button type="button" class="btn btn-primary btn-copy-ex" aria-label="Copy to clipboard" data-toggle="tooltip" data-placement="left" data-trigger="hover" data-clipboard-copy="" data-bs-original-title="Copy to clipboard"><i class="fa fa-copy"></i></button><pre class="downlit sourceCode r"><code class="sourceCode R"><span><span class="va">weather2</span> <span class="op">&lt;-</span> <span class="va">weather</span> <span class="op"><a href="../reference/pipe.html">%&gt;%</a></span> </span>
<span>  <span class="fu"><a href="../reference/pivot_longer.html">pivot_longer</a></span><span class="op">(</span></span>
<span>    <span class="va">d1</span><span class="op">:</span><span class="va">d31</span>, </span>
<span>    names_to <span class="op">=</span> <span class="st">"day"</span>, </span>
<span>    values_to <span class="op">=</span> <span class="st">"value"</span>, </span>
<span>    values_drop_na <span class="op">=</span> <span class="cn">TRUE</span></span>
<span>  <span class="op">)</span> </span>
<span><span class="va">weather2</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># A tibble: 66 × 6</span></span></span>
<span><span class="co">#&gt;    id       year month element day   value</span></span>
<span><span class="co">#&gt;    <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>   <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>   <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 1</span> MX17004  <span style="text-decoration: underline;">2</span>010     1 tmax    d30    27.8</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 2</span> MX17004  <span style="text-decoration: underline;">2</span>010     1 tmin    d30    14.5</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 3</span> MX17004  <span style="text-decoration: underline;">2</span>010     2 tmax    d2     27.3</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 4</span> MX17004  <span style="text-decoration: underline;">2</span>010     2 tmax    d3     24.1</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 5</span> MX17004  <span style="text-decoration: underline;">2</span>010     2 tmax    d11    29.7</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 6</span> MX17004  <span style="text-decoration: underline;">2</span>010     2 tmax    d23    29.9</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 7</span> MX17004  <span style="text-decoration: underline;">2</span>010     2 tmin    d2     14.4</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 8</span> MX17004  <span style="text-decoration: underline;">2</span>010     2 tmin    d3     14.4</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 9</span> MX17004  <span style="text-decoration: underline;">2</span>010     2 tmin    d11    13.4</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;">10</span> MX17004  <span style="text-decoration: underline;">2</span>010     2 tmin    d23    10.7</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 56 more rows</span></span></span></code></pre></div>

<p>For presentation, I’ve dropped the missing values, making them implicit rather than explicit. This is ok because we know how many days are in each month and can easily reconstruct the explicit missing values.</p>

<p>We’ll also do a little cleaning:</p>

<div class="sourceCode hasCopyButton" id="cb18"><button type="button" class="btn btn-primary btn-copy-ex" aria-label="Copy to clipboard" data-toggle="tooltip" data-placement="left" data-trigger="hover" data-clipboard-copy="" data-bs-original-title="Copy to clipboard"><i class="fa fa-copy"></i></button><pre class="downlit sourceCode r"><code class="sourceCode R"><span><span class="va">weather3</span> <span class="op">&lt;-</span> <span class="va">weather2</span> <span class="op"><a href="../reference/pipe.html">%&gt;%</a></span> </span>
<span>  <span class="fu"><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate</a></span><span class="op">(</span>day <span class="op">=</span> <span class="fu"><a href="https://rdrr.io/r/base/integer.html">as.integer</a></span><span class="op">(</span><span class="fu"><a href="https://rdrr.io/r/base/grep.html">gsub</a></span><span class="op">(</span><span class="st">"d"</span>, <span class="st">""</span>, <span class="va">day</span><span class="op">)</span><span class="op">)</span><span class="op">)</span> <span class="op"><a href="../reference/pipe.html">%&gt;%</a></span></span>
<span>  <span class="fu"><a href="https://dplyr.tidyverse.org/reference/select.html">select</a></span><span class="op">(</span><span class="va">id</span>, <span class="va">year</span>, <span class="va">month</span>, <span class="va">day</span>, <span class="va">element</span>, <span class="va">value</span><span class="op">)</span></span>
<span><span class="va">weather3</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># A tibble: 66 × 6</span></span></span>
<span><span class="co">#&gt;    id       year month   day element value</span></span>
<span><span class="co">#&gt;    <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>   <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>   <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 1</span> MX17004  <span style="text-decoration: underline;">2</span>010     1    30 tmax     27.8</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 2</span> MX17004  <span style="text-decoration: underline;">2</span>010     1    30 tmin     14.5</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 3</span> MX17004  <span style="text-decoration: underline;">2</span>010     2     2 tmax     27.3</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 4</span> MX17004  <span style="text-decoration: underline;">2</span>010     2     3 tmax     24.1</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 5</span> MX17004  <span style="text-decoration: underline;">2</span>010     2    11 tmax     29.7</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 6</span> MX17004  <span style="text-decoration: underline;">2</span>010     2    23 tmax     29.9</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 7</span> MX17004  <span style="text-decoration: underline;">2</span>010     2     2 tmin     14.4</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 8</span> MX17004  <span style="text-decoration: underline;">2</span>010     2     3 tmin     14.4</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 9</span> MX17004  <span style="text-decoration: underline;">2</span>010     2    11 tmin     13.4</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;">10</span> MX17004  <span style="text-decoration: underline;">2</span>010     2    23 tmin     10.7</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 56 more rows</span></span></span></code></pre></div>

<p>This dataset is mostly tidy, but the <code>element</code> column is not a variable; it stores the names of variables. (Not shown in this example are the other meteorological variables <code>prcp</code> (precipitation) and <code>snow</code> (snowfall)). Fixing this requires widening the data: <code><a href="../reference/pivot_wider.html">pivot_wider()</a></code> is inverse of <code><a href="../reference/pivot_longer.html">pivot_longer()</a></code>, pivoting <code>element</code> and <code>value</code> back out across multiple columns:</p>

<div class="sourceCode hasCopyButton" id="cb19"><button type="button" class="btn btn-primary btn-copy-ex" aria-label="Copy to clipboard" data-toggle="tooltip" data-placement="left" data-trigger="hover" data-clipboard-copy="" data-bs-original-title="Copy to clipboard"><i class="fa fa-copy"></i></button><pre class="downlit sourceCode r"><code class="sourceCode R"><span><span class="va">weather3</span> <span class="op"><a href="../reference/pipe.html">%&gt;%</a></span> </span>
<span>  <span class="fu"><a href="../reference/pivot_wider.html">pivot_wider</a></span><span class="op">(</span>names_from <span class="op">=</span> <span class="va">element</span>, values_from <span class="op">=</span> <span class="va">value</span><span class="op">)</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># A tibble: 33 × 6</span></span></span>
<span><span class="co">#&gt;    id       year month   day  tmax  tmin</span></span>
<span><span class="co">#&gt;    <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>   <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 1</span> MX17004  <span style="text-decoration: underline;">2</span>010     1    30  27.8  14.5</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 2</span> MX17004  <span style="text-decoration: underline;">2</span>010     2     2  27.3  14.4</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 3</span> MX17004  <span style="text-decoration: underline;">2</span>010     2     3  24.1  14.4</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 4</span> MX17004  <span style="text-decoration: underline;">2</span>010     2    11  29.7  13.4</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 5</span> MX17004  <span style="text-decoration: underline;">2</span>010     2    23  29.9  10.7</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 6</span> MX17004  <span style="text-decoration: underline;">2</span>010     3     5  32.1  14.2</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 7</span> MX17004  <span style="text-decoration: underline;">2</span>010     3    10  34.5  16.8</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 8</span> MX17004  <span style="text-decoration: underline;">2</span>010     3    16  31.1  17.6</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 9</span> MX17004  <span style="text-decoration: underline;">2</span>010     4    27  36.3  16.7</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;">10</span> MX17004  <span style="text-decoration: underline;">2</span>010     5    27  33.2  18.2</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 23 more rows</span></span></span></code></pre></div>

<p>This form is tidy: there’s one variable in each column, and each row represents one day.</p>

## <h3 id="multiple-types">Multiple types in one table</h3>

<p>Datasets often involve values collected at multiple levels, on different types of observational units. During tidying, each type of observational unit should be stored in its own table. This is closely related to the idea of database normalisation, where each fact is expressed in only one place. It’s important because otherwise inconsistencies can arise.</p>

<p>The billboard dataset actually contains observations on two types of observational units: the song and its rank in each week. This manifests itself through the duplication of facts about the song: <code>artist</code> is repeated many times.</p>

<p>This dataset needs to be broken down into two pieces: a song dataset which stores <code>artist</code> and <code>song name</code>, and a ranking dataset which gives the <code>rank</code> of the <code>song</code> in each <code>week</code>. We first extract a <code>song</code> dataset:</p>

<div class="sourceCode hasCopyButton" id="cb20"><button type="button" class="btn btn-primary btn-copy-ex" aria-label="Copy to clipboard" data-toggle="tooltip" data-placement="left" data-trigger="hover" data-clipboard-copy="" data-bs-original-title="Copy to clipboard"><i class="fa fa-copy"></i></button><pre class="downlit sourceCode r"><code class="sourceCode R"><span><span class="va">song</span> <span class="op">&lt;-</span> <span class="va">billboard3</span> <span class="op"><a href="../reference/pipe.html">%&gt;%</a></span> </span>
<span>  <span class="fu"><a href="https://dplyr.tidyverse.org/reference/distinct.html">distinct</a></span><span class="op">(</span><span class="va">artist</span>, <span class="va">track</span><span class="op">)</span> <span class="op"><a href="../reference/pipe.html">%&gt;%</a></span></span>
<span>  <span class="fu"><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate</a></span><span class="op">(</span>song_id <span class="op">=</span> <span class="fu"><a href="https://dplyr.tidyverse.org/reference/row_number.html">row_number</a></span><span class="op">(</span><span class="op">)</span><span class="op">)</span></span>
<span><span class="va">song</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># A tibble: 317 × 3</span></span></span>
<span><span class="co">#&gt;    artist         track                   song_id</span></span>
<span><span class="co">#&gt;    <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>          <span style="color: #949494; font-style: italic;">&lt;chr&gt;</span>                     <span style="color: #949494; font-style: italic;">&lt;int&gt;</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 1</span> 2 Pac          Baby Don't Cry (Keep...       1</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 2</span> 2Ge+her        The Hardest Part Of ...       2</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 3</span> 3 Doors Down   Kryptonite                    3</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 4</span> 3 Doors Down   Loser                         4</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 5</span> 504 Boyz       Wobble Wobble                 5</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 6</span> 98^0           Give Me Just One Nig...       6</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 7</span> A*Teens        Dancing Queen                 7</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 8</span> Aaliyah        I Don't Wanna                 8</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 9</span> Aaliyah        Try Again                     9</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;">10</span> Adams, Yolanda Open My Heart                10</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 307 more rows</span></span></span></code></pre></div>

<p>Then use that to make a <code>rank</code> dataset by replacing repeated song facts with a pointer to song details (a unique song id):</p>

<div class="sourceCode hasCopyButton" id="cb21"><button type="button" class="btn btn-primary btn-copy-ex" aria-label="Copy to clipboard" data-toggle="tooltip" data-placement="left" data-trigger="hover" data-clipboard-copy="" data-bs-original-title="Copy to clipboard"><i class="fa fa-copy"></i></button><pre class="downlit sourceCode r"><code class="sourceCode R"><span><span class="va">rank</span> <span class="op">&lt;-</span> <span class="va">billboard3</span> <span class="op"><a href="../reference/pipe.html">%&gt;%</a></span></span>
<span>  <span class="fu"><a href="https://dplyr.tidyverse.org/reference/mutate-joins.html">left_join</a></span><span class="op">(</span><span class="va">song</span>, <span class="fu"><a href="https://rdrr.io/r/base/c.html">c</a></span><span class="op">(</span><span class="st">"artist"</span>, <span class="st">"track"</span><span class="op">)</span><span class="op">)</span> <span class="op"><a href="../reference/pipe.html">%&gt;%</a></span></span>
<span>  <span class="fu"><a href="https://dplyr.tidyverse.org/reference/select.html">select</a></span><span class="op">(</span><span class="va">song_id</span>, <span class="va">date</span>, <span class="va">week</span>, <span class="va">rank</span><span class="op">)</span></span>
<span><span class="va">rank</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># A tibble: 5,307 × 4</span></span></span>
<span><span class="co">#&gt;    song_id date        week  rank</span></span>
<span><span class="co">#&gt;      <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;date&gt;</span>     <span style="color: #949494; font-style: italic;">&lt;int&gt;</span> <span style="color: #949494; font-style: italic;">&lt;dbl&gt;</span></span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 1</span>       1 2000-02-26     1    87</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 2</span>       1 2000-03-04     2    82</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 3</span>       1 2000-03-11     3    72</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 4</span>       1 2000-03-18     4    77</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 5</span>       1 2000-03-25     5    87</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 6</span>       1 2000-04-01     6    94</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 7</span>       1 2000-04-08     7    99</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 8</span>       2 2000-09-02     1    91</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;"> 9</span>       2 2000-09-09     2    87</span></span>
<span><span class="co">#&gt; <span style="color: #BCBCBC;">10</span>       2 2000-09-16     3    92</span></span>
<span><span class="co">#&gt; <span style="color: #949494;"># ℹ 5,297 more rows</span></span></span></code></pre></div>

<p>You could also imagine a <code>week</code> dataset which would record background information about the week, maybe the total number of songs sold or similar “demographic” information.</p>

<p>Normalisation is useful for tidying and eliminating inconsistencies. However, there are few data analysis tools that work directly with relational data, so analysis usually also requires denormalisation or the merging the datasets back into one table.</p>

## <h3 id="one-type-in-multiple-tables">One type in multiple tables</h3>

<p>It’s also common to find data values about a single type of observational unit spread out over multiple tables or files. These tables and files are often split up by another variable, so that each represents a single year, person, or location. As long as the format for individual records is consistent, this is an easy problem to fix:</p>

<ol>
<li><p>Read the files into a list of tables.</p></li>
<li><p>For each table, add a new column that records the original file name (the file name is often the value of an important variable).</p></li>
<li><p>Combine all tables into a single table.</p></li>
</ol>

<p>Purrr makes this straightforward in R. The following code generates a vector of file names in a directory (<code>data/</code>) which match a regular expression (ends in <code>.csv</code>). Next we name each element of the vector with the name of the file. We do this because will preserve the names in the following step, ensuring that each row in the final data frame is labeled with its source. Finally, <code><a href="https://purrr.tidyverse.org/reference/map_dfr.html">map_dfr()</a></code> loops over each path, reading in the csv file and
combining the results into a single data frame.</p>

<div class="sourceCode hasCopyButton" id="cb22"><button type="button" class="btn btn-primary btn-copy-ex" aria-label="Copy to clipboard" data-toggle="tooltip" data-placement="left" data-trigger="hover" data-clipboard-copy="" data-bs-original-title="Copy to clipboard"><i class="fa fa-copy"></i></button><pre class="downlit sourceCode r"><code class="sourceCode R"><span><span class="kw"><a href="https://rdrr.io/r/base/library.html">library</a></span><span class="op">(</span><span class="va"><a href="https://purrr.tidyverse.org/">purrr</a></span><span class="op">)</span></span>
<span><span class="va">paths</span> <span class="op">&lt;-</span> <span class="fu"><a href="https://rdrr.io/r/base/list.files.html">dir</a></span><span class="op">(</span><span class="st">"data"</span>, pattern <span class="op">=</span> <span class="st">"\\.csv$"</span>, full.names <span class="op">=</span> <span class="cn">TRUE</span><span class="op">)</span></span>
<span><span class="fu"><a href="https://rdrr.io/r/base/names.html">names</a></span><span class="op">(</span><span class="va">paths</span><span class="op">)</span> <span class="op">&lt;-</span> <span class="fu"><a href="https://rdrr.io/r/base/basename.html">basename</a></span><span class="op">(</span><span class="va">paths</span><span class="op">)</span></span>
<span><span class="fu"><a href="https://purrr.tidyverse.org/reference/map_dfr.html">map_dfr</a></span><span class="op">(</span><span class="va">paths</span>, <span class="va">read.csv</span>, stringsAsFactors <span class="op">=</span> <span class="cn">FALSE</span>, .id <span class="op">=</span> <span class="st">"filename"</span><span class="op">)</span></span></code></pre></div>

<p>Once you have a single table, you can perform additional tidying as needed. An example of this type of cleaning can be found at <a href="https://github.com/hadley/data-baby-names">https://github.com/hadley/data-baby-names</a> which takes 129 yearly baby name tables provided by the US Social Security Administration and combines them into a single file.</p>

<p>A more complicated situation occurs when the dataset structure changes over time. For example, the datasets may contain different variables, the same variables with different names, different file formats, or different conventions for missing values. This may require you to tidy each file to individually (or, if you’re lucky, in small groups) and then combine them once tidied. An example of this type of tidying is illustrated in <a href="https://github.com/hadley/data-fuel-economy">https://github.com/hadley/data-fuel-economy</a>, which shows the tidying of <span>epa</span> fuel economy data for over 50,000 cars from 1978 to 2008. The raw data is available online, but each year is stored in a separate file and there are four major formats with many minor variations, making tidying this dataset a considerable challenge.</p>